# MIMIC-IV Demo: linked canonical ICU vital-sign resource

This notebook defines the **unFAIRified source resource** used in the FAIRification project. It starts from the **MIMIC-IV Clinical Database Demo v2.2** files stored locally under `data/` and derives a bounded, reproducible resource around three ICU vital signs:

- Heart rate (`itemid` 220045)
- Respiratory rate (`itemid` 220210)
- Pulse-oximetry oxygen saturation (`itemid` 220277)

The observation scope remains the **first four hours of every represented ICU stay**. The four-hour window is an engineering scope boundary for the course project, not a clinical claim about the importance of the first four ICU hours.

The canonical resource is now a **linked bundle of five tabular files**, each kept at its natural entity grain. This avoids flattening transfer intervals and admission diagnoses onto observation rows. The files contain only source data plus the explicit `stay_id` linkage derived for care-unit intervals. **No FHIR structure or external terminology mapping (LOINC, UCUM, SNOMED CT, canonical ICD formatting) is introduced here.** Those belong to the later FAIRification transformation.


## 1. Setup and source files

Seven source tables are required:

- `icu/chartevents.csv.gz` — ICU observations
- `icu/icustays.csv.gz` — ICU stay identifiers, periods, and source first/last care-unit summaries used for validation
- `icu/d_items.csv.gz` — dictionary for MIMIC `itemid` values
- `hosp/transfers.csv.gz` — time-resolved hospital/ICU location intervals
- `hosp/admissions.csv.gz` — hospital admission periods and admission types
- `hosp/diagnoses_icd.csv.gz` — encounter diagnosis codes and billing rank
- `hosp/d_icd_diagnoses.csv.gz` — MIMIC diagnosis-code dictionary

The helper below locates the repository root whether the notebook is started from the project root or from `notebooks/`.


In [1]:
from pathlib import Path
import hashlib

import pandas as pd

pd.set_option("display.max_columns", 30)


def find_project_root() -> Path:
    """Find the repository root by looking for the expected MIMIC directories."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "icu").is_dir() and (candidate / "data" / "hosp").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/icu/ and data/hosp/. Run this notebook from inside the "
        "project repository and place the MIMIC-IV Demo files under those directories."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ICU_DIR = DATA_DIR / "icu"
HOSP_DIR = DATA_DIR / "hosp"
DERIVED_DIR = DATA_DIR / "derived" / "mimic_icu_vitals_4h"

CHARTEVENTS_PATH = ICU_DIR / "chartevents.csv.gz"
ICUSTAYS_PATH = ICU_DIR / "icustays.csv.gz"
D_ITEMS_PATH = ICU_DIR / "d_items.csv.gz"
ADMISSIONS_PATH = HOSP_DIR / "admissions.csv.gz"
TRANSFERS_PATH = HOSP_DIR / "transfers.csv.gz"
DIAGNOSES_PATH = HOSP_DIR / "diagnoses_icd.csv.gz"
D_ICD_PATH = HOSP_DIR / "d_icd_diagnoses.csv.gz"

TARGET_ITEMIDS = [220045, 220210, 220277]
WINDOW_HOURS = 4

source_paths = [
    CHARTEVENTS_PATH,
    ICUSTAYS_PATH,
    D_ITEMS_PATH,
    ADMISSIONS_PATH,
    TRANSFERS_PATH,
    DIAGNOSES_PATH,
    D_ICD_PATH,
]

print(f"Project root: {PROJECT_ROOT}")
for path in source_paths:
    print(f"Source exists: {path.relative_to(PROJECT_ROOT)} -> {path.is_file()}")


Project root: /mnt/data/fair_rebuild
Source exists: data/icu/chartevents.csv.gz -> True
Source exists: data/icu/icustays.csv.gz -> True
Source exists: data/icu/d_items.csv.gz -> True
Source exists: data/hosp/admissions.csv.gz -> True
Source exists: data/hosp/transfers.csv.gz -> True
Source exists: data/hosp/diagnoses_icd.csv.gz -> True
Source exists: data/hosp/d_icd_diagnoses.csv.gz -> True


## 2. Load only the source columns required by the bounded resource

The source tables are much wider than the project resource. We load only fields needed to select the cohort, preserve the relevant source values, construct the five linked tables, and validate the transfer history.


In [2]:
chartevents = pd.read_csv(
    CHARTEVENTS_PATH,
    dtype={"warning": "Int64"},
    usecols=[
        "subject_id", "hadm_id", "stay_id", "charttime", "itemid",
        "valuenum", "valueuom", "warning",
    ],
)

icustays = pd.read_csv(
    ICUSTAYS_PATH,
    usecols=[
        "subject_id", "hadm_id", "stay_id", "first_careunit", "last_careunit",
        "intime", "outtime",
    ],
)

d_items = pd.read_csv(
    D_ITEMS_PATH,
    usecols=["itemid", "label", "abbreviation", "category", "unitname", "param_type"],
)

admissions = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "admission_type"],
)

transfers = pd.read_csv(
    TRANSFERS_PATH,
    dtype={"hadm_id": "Int64"},
    usecols=["subject_id", "hadm_id", "transfer_id", "eventtype", "careunit", "intime", "outtime"],
)

diagnoses_icd = pd.read_csv(
    DIAGNOSES_PATH,
    dtype={"icd_code": "string"},
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
)

d_icd_diagnoses = pd.read_csv(
    D_ICD_PATH,
    dtype={"icd_code": "string"},
    usecols=["icd_code", "icd_version", "long_title"],
)

source_summary = pd.DataFrame({
    "table": [
        "chartevents", "icustays", "d_items", "admissions", "transfers",
        "diagnoses_icd", "d_icd_diagnoses",
    ],
    "rows_loaded": [
        len(chartevents), len(icustays), len(d_items), len(admissions), len(transfers),
        len(diagnoses_icd), len(d_icd_diagnoses),
    ],
    "columns_loaded": [
        chartevents.shape[1], icustays.shape[1], d_items.shape[1], admissions.shape[1],
        transfers.shape[1], diagnoses_icd.shape[1], d_icd_diagnoses.shape[1],
    ],
})
source_summary


,table,rows_loaded,columns_loaded
0,chartevents,668862,8
1,icustays,140,7
2,d_items,4014,6
3,admissions,275,5
4,transfers,1190,7
5,diagnoses_icd,4506,5
6,d_icd_diagnoses,109775,3


## 3. Confirm the three source measurement definitions

MIMIC uses local integer `itemid` values for ICU charted measurements. The dictionary is retained as source evidence for what the three selected measurements mean. External terminology mappings are deliberately deferred to the FAIRification stage.


In [3]:
target_items = (
    d_items[d_items["itemid"].isin(TARGET_ITEMIDS)]
    .sort_values("itemid")
    .reset_index(drop=True)
)
target_items


,itemid,label,abbreviation,category,unitname,param_type
0,220045,Heart Rate,HR,Routine Vital Signs,bpm,Numeric
1,220210,Respiratory Rate,RR,Respiratory,insp/min,Numeric
2,220277,O2 saturation pulseoxymetry,SpO2,Respiratory,%,Numeric


## 4. Select the first four hours of each ICU stay

The window is defined relative to `icustays.intime`, not by clock time. The interval is half-open: `[ICU intime, ICU intime + 4 hours)`.

`charttime` is the source observation time. The resulting `observations.csv` retains the MIMIC identifiers, source measurement label, numeric value, source unit string, and warning flag.


In [4]:
for frame, columns in [
    (chartevents, ["charttime"]),
    (icustays, ["intime", "outtime"]),
    (admissions, ["admittime", "dischtime"]),
    (transfers, ["intime", "outtime"]),
]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])

selected_events = chartevents[chartevents["itemid"].isin(TARGET_ITEMIDS)].copy()
assert icustays["stay_id"].is_unique

selected_events = selected_events.merge(
    icustays[["subject_id", "hadm_id", "stay_id", "intime"]],
    on=["subject_id", "hadm_id", "stay_id"],
    how="left",
    validate="many_to_one",
)
assert selected_events["intime"].notna().all(), "An observation has no matching ICU stay."

window_end = selected_events["intime"] + pd.Timedelta(hours=WINDOW_HOURS)
observations = selected_events[
    (selected_events["charttime"] >= selected_events["intime"])
    & (selected_events["charttime"] < window_end)
].copy()

observations = observations.merge(
    d_items[["itemid", "label"]],
    on="itemid",
    how="left",
    validate="many_to_one",
).rename(columns={"label": "measurement_label"})

observations = observations[
    [
        "subject_id", "hadm_id", "stay_id", "charttime", "itemid",
        "measurement_label", "valuenum", "valueuom", "warning",
    ]
].sort_values(
    ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum"],
    kind="stable",
).reset_index(drop=True)

observations.head(10)


,subject_id,hadm_id,stay_id,charttime,itemid,measurement_label,valuenum,valueuom,warning
0,10000032,29079034,39553978,2180-07-23 14:12:00,220045,Heart Rate,91.0,bpm,0
1,10000032,29079034,39553978,2180-07-23 14:12:00,220210,Respiratory Rate,24.0,insp/min,0
2,10000032,29079034,39553978,2180-07-23 14:13:00,220277,O2 saturation pulseoxymetry,98.0,%,0
3,10000032,29079034,39553978,2180-07-23 14:30:00,220045,Heart Rate,93.0,bpm,0
4,10000032,29079034,39553978,2180-07-23 14:30:00,220210,Respiratory Rate,21.0,insp/min,0
5,10000032,29079034,39553978,2180-07-23 14:30:00,220277,O2 saturation pulseoxymetry,97.0,%,0
6,10000032,29079034,39553978,2180-07-23 15:00:00,220045,Heart Rate,94.0,bpm,0
7,10000032,29079034,39553978,2180-07-23 15:00:00,220210,Respiratory Rate,23.0,insp/min,0
8,10000032,29079034,39553978,2180-07-23 15:00:00,220277,O2 saturation pulseoxymetry,97.0,%,0
9,10000032,29079034,39553978,2180-07-23 16:00:00,220045,Heart Rate,105.0,bpm,0


## 5. Derive the linked ICU-stay and care-unit interval tables

`icu_stays.csv` contains one row for each ICU stay represented by the observation subset.

For `careunit_intervals.csv`, source `transfers` rows with a non-null `careunit` are matched to the represented ICU stay whose interval contains the transfer interval. The added `stay_id` is therefore a **derived linkage field**; the remaining fields come directly from `transfers`.

The source `first_careunit` and `last_careunit` fields are not carried into the canonical output. Instead, they are used below to validate that the time-resolved transfer history reproduces the source ICU-stay summaries.


In [5]:
represented_stay_ids = observations["stay_id"].drop_duplicates()

represented_icustays = icustays[icustays["stay_id"].isin(represented_stay_ids)].copy()
assert represented_icustays["stay_id"].is_unique

icu_stays = represented_icustays[
    ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
].sort_values(["subject_id", "hadm_id", "stay_id"], kind="stable").reset_index(drop=True)

# Match care-unit transfer intervals to the single represented ICU stay that fully contains them.
transfer_candidates = transfers[
    transfers["hadm_id"].isin(observations["hadm_id"].unique())
    & transfers["careunit"].notna()
].copy()

matched = transfer_candidates.merge(
    represented_icustays[["subject_id", "hadm_id", "stay_id", "intime", "outtime"]],
    on=["subject_id", "hadm_id"],
    how="inner",
    suffixes=("_transfer", "_stay"),
)
matched = matched[
    (matched["intime_transfer"] >= matched["intime_stay"])
    & (matched["outtime_transfer"] <= matched["outtime_stay"])
].copy()

# A source transfer interval must link to at most one ICU stay in this bounded cohort.
assert not matched["transfer_id"].duplicated().any(), "A transfer interval matched multiple ICU stays."

careunit_intervals = matched[
    [
        "subject_id", "hadm_id", "stay_id", "transfer_id", "eventtype", "careunit",
        "intime_transfer", "outtime_transfer",
    ]
].rename(columns={
    "intime_transfer": "intime",
    "outtime_transfer": "outtime",
}).sort_values(
    ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "transfer_id"],
    kind="stable",
).reset_index(drop=True)

careunit_intervals.head(10)


,subject_id,hadm_id,stay_id,transfer_id,eventtype,careunit,intime,outtime
0,10000032,29079034,39553978,39553978,admit,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47
1,10001217,24597018,37067082,37067082,transfer,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00
2,10001217,27703517,34592300,34592300,transfer,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41
3,10001725,25563031,31205490,31205490,transfer,Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56
4,10002428,20321825,34807493,34807493,admit,Medical Intensive Care Unit (MICU),2156-04-30 21:53:00,2156-05-02 22:27:20
5,10002428,23473524,35479615,35479615,admit,Surgical Intensive Care Unit (SICU),2156-05-11 14:49:34,2156-05-11 14:52:30
6,10002428,23473524,35479615,39816752,transfer,Surgical Intensive Care Unit (SICU),2156-05-11 14:52:30,2156-05-15 20:23:28
7,10002428,23473524,35479615,30896594,transfer,Medical Intensive Care Unit (MICU),2156-05-15 20:23:28,2156-05-22 14:16:46
8,10002428,28662225,33987268,33987268,transfer,Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08
9,10002428,28662225,38875437,38875437,transfer,Medical Intensive Care Unit (MICU),2156-04-19 18:11:19,2156-04-19 18:13:35


## 6. Derive hospital-admission and first-ranked diagnosis tables

`hospital_admissions.csv` contains the 121 hospital admissions represented by the ICU observation cohort.

`first_ranked_diagnoses.csv` contains only `seq_num = 1` from `diagnoses_icd`, with the source dictionary title joined from `d_icd_diagnoses`. This preserves MIMIC's original ICD code string and version marker. The table does **not** reinterpret `seq_num = 1` as a biological cause or normalize the ICD code; those semantic decisions belong to the later model/transformation.


In [6]:
represented_hadm_ids = observations["hadm_id"].drop_duplicates()

hospital_admissions = admissions[
    admissions["hadm_id"].isin(represented_hadm_ids)
][
    ["subject_id", "hadm_id", "admittime", "dischtime", "admission_type"]
].sort_values(["subject_id", "hadm_id"], kind="stable").reset_index(drop=True)

first_ranked_diagnoses = diagnoses_icd[
    diagnoses_icd["hadm_id"].isin(represented_hadm_ids)
    & (diagnoses_icd["seq_num"] == 1)
].copy()

first_ranked_diagnoses = first_ranked_diagnoses.merge(
    d_icd_diagnoses,
    on=["icd_code", "icd_version"],
    how="left",
    validate="many_to_one",
)

first_ranked_diagnoses = first_ranked_diagnoses[
    ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version", "long_title"]
].sort_values(["subject_id", "hadm_id"], kind="stable").reset_index(drop=True)

hospital_admissions.head(), first_ranked_diagnoses.head()


(   subject_id   hadm_id           admittime           dischtime admission_type
 0    10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00       EW EMER.
 1    10001217  24597018 2157-11-18 22:56:00 2157-11-25 18:00:00       EW EMER.
 2    10001217  27703517 2157-12-18 16:58:00 2157-12-24 14:55:00   DIRECT EMER.
 3    10001725  25563031 2110-04-11 15:08:00 2110-04-14 15:00:00       EW EMER.
 4    10002428  20321825 2156-04-30 20:35:00 2156-05-03 16:36:00       EW EMER.,
    subject_id   hadm_id  seq_num icd_code  icd_version  \
 0    10000032  29079034        1    45829            9   
 1    10001217  24597018        1     3240            9   
 2    10001217  27703517        1     3240            9   
 3    10001725  25563031        1    78829            9   
 4    10002428  20321825        1    51881            9   
 
                            long_title  
 0        Other iatrogenic hypotension  
 1                Intracranial abscess  
 2                Intracranial abscess  

## 7. Reproducibility and relational-integrity checks

These checks make changes in the source release or derivation logic visible rather than silently producing a different canonical resource.


In [7]:
# Observation invariants.
expected_item_counts = {220045: 859, 220210: 860, 220277: 782}
assert len(observations) == 2501
assert observations["subject_id"].nunique() == 94
assert observations["hadm_id"].nunique() == 121
assert observations["stay_id"].nunique() == 133
assert observations["itemid"].value_counts().sort_index().to_dict() == expected_item_counts
assert observations[["measurement_label", "valuenum", "valueuom", "warning"]].notna().all().all()
assert observations["warning"].value_counts().to_dict() == {0: 2500, 1: 1}
assert not observations.duplicated(["stay_id", "charttime", "itemid"]).any()

# ICU-stay invariants.
assert len(icu_stays) == 133
assert icu_stays["stay_id"].is_unique
assert set(icu_stays["stay_id"]) == set(observations["stay_id"])
assert (icu_stays["intime"] < icu_stays["outtime"]).all()

# Care-unit interval invariants.
assert len(careunit_intervals) == 167
assert careunit_intervals["stay_id"].nunique() == 133
assert careunit_intervals["careunit"].nunique() == 9
assert careunit_intervals.groupby("stay_id").size().value_counts().sort_index().to_dict() == {1: 105, 2: 22, 3: 6}

care_validation = careunit_intervals.merge(
    represented_icustays[["stay_id", "intime", "outtime", "first_careunit", "last_careunit"]],
    on="stay_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_stay"),
)
assert (care_validation["intime"] >= care_validation["intime_stay"]).all()
assert (care_validation["outtime"] <= care_validation["outtime_stay"]).all()

ordered_care = care_validation.sort_values(["stay_id", "intime", "outtime", "transfer_id"], kind="stable")
first_intervals = ordered_care.groupby("stay_id", sort=False).first()
last_intervals = ordered_care.groupby("stay_id", sort=False).last()
assert (first_intervals["intime"] == first_intervals["intime_stay"]).all()
assert (last_intervals["outtime"] == last_intervals["outtime_stay"]).all()
assert (first_intervals["careunit"] == first_intervals["first_careunit"]).all()
assert (last_intervals["careunit"] == last_intervals["last_careunit"]).all()

for _, group in careunit_intervals.groupby("stay_id"):
    group = group.sort_values(["intime", "outtime", "transfer_id"], kind="stable")
    if len(group) > 1:
        assert (group["outtime"].iloc[:-1].reset_index(drop=True) == group["intime"].iloc[1:].reset_index(drop=True)).all()

# Hospital-admission invariants.
assert len(hospital_admissions) == 121
assert hospital_admissions["hadm_id"].is_unique
assert set(hospital_admissions["hadm_id"]) == set(observations["hadm_id"])
assert (hospital_admissions["admittime"] < hospital_admissions["dischtime"]).all()
assert hospital_admissions["admission_type"].nunique() == 6

# Diagnosis invariants.
assert len(first_ranked_diagnoses) == 121
assert first_ranked_diagnoses["hadm_id"].is_unique
assert set(first_ranked_diagnoses["hadm_id"]) == set(observations["hadm_id"])
assert first_ranked_diagnoses["seq_num"].eq(1).all()
assert first_ranked_diagnoses["long_title"].notna().all()
assert first_ranked_diagnoses["icd_version"].value_counts().to_dict() == {9: 72, 10: 49}
assert first_ranked_diagnoses[["icd_code", "icd_version"]].drop_duplicates().shape[0] == 99

print("All reproducibility and relational-integrity checks passed.")


All reproducibility and relational-integrity checks passed.


## 8. Inspect the bounded linked resource

These summaries show the natural grain of each component before any semantic enrichment is applied.


In [8]:
resource_summary = pd.DataFrame([
    {
        "file": "observations.csv",
        "rows": len(observations),
        "grain": "one selected charted vital-sign observation",
    },
    {
        "file": "icu_stays.csv",
        "rows": len(icu_stays),
        "grain": "one represented ICU stay",
    },
    {
        "file": "careunit_intervals.csv",
        "rows": len(careunit_intervals),
        "grain": "one time-bounded care-unit transfer interval within a represented ICU stay",
    },
    {
        "file": "hospital_admissions.csv",
        "rows": len(hospital_admissions),
        "grain": "one represented hospital admission",
    },
    {
        "file": "first_ranked_diagnoses.csv",
        "rows": len(first_ranked_diagnoses),
        "grain": "the source seq_num=1 diagnosis for one represented hospital admission",
    },
])
resource_summary


,file,rows,grain
0,observations.csv,2501,one selected charted vital-sign observation
1,icu_stays.csv,133,one represented ICU stay
2,careunit_intervals.csv,167,one time-bounded care-unit transfer interval w...
3,hospital_admissions.csv,121,one represented hospital admission
4,first_ranked_diagnoses.csv,121,the source seq_num=1 diagnosis for one represe...


In [9]:
careunit_intervals["careunit"].value_counts().rename("intervals").to_frame()


,intervals
careunit,
Medical Intensive Care Unit (MICU),36
Surgical Intensive Care Unit (SICU),33
Medical/Surgical Intensive Care Unit (MICU/SICU),31
Cardiac Vascular Intensive Care Unit (CVICU),25
Trauma SICU (TSICU),20
Coronary Care Unit (CCU),16
Neuro Stepdown,3
Neuro Surgical Intensive Care Unit (Neuro SICU),2
Neuro Intermediate,1


In [10]:
first_ranked_diagnoses["icd_version"].value_counts().sort_index().rename("diagnoses").to_frame()


,diagnoses
icd_version,
9,72
10,49


## 9. Save the canonical linked resource

The output directory contains the five tabular components of the bounded **unFAIRified** project resource. The original MIMIC-IV Demo remains the provenance source. The source identifiers and source code strings are preserved; external semantic mappings are intentionally absent.


In [11]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "observations.csv": observations,
    "icu_stays.csv": icu_stays,
    "careunit_intervals.csv": careunit_intervals,
    "hospital_admissions.csv": hospital_admissions,
    "first_ranked_diagnoses.csv": first_ranked_diagnoses,
}

for filename, frame in outputs.items():
    path = DERIVED_DIR / filename
    frame.to_csv(path, index=False, date_format="%Y-%m-%d %H:%M:%S")

hash_rows = []
for filename in outputs:
    path = DERIVED_DIR / filename
    hash_rows.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(outputs[filename]),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })

hash_summary = pd.DataFrame(hash_rows)
hash_summary


,file,rows,sha256
0,data/derived/mimic_icu_vitals_4h/observations.csv,2501,22cc760c9f731de423d0022abf00022c78c9f0be091cc6...
1,data/derived/mimic_icu_vitals_4h/icu_stays.csv,133,0018c771c9b700cf8d726549b144884537f2ec1565bdd9...
2,data/derived/mimic_icu_vitals_4h/careunit_inte...,167,bfa961c9463edfca9c2a5bc0070746f7a729e7659cb1b1...
3,data/derived/mimic_icu_vitals_4h/hospital_admi...,121,0f29a8c07aa419eedd780af5bf2f09aefc18d3bb21185d...
4,data/derived/mimic_icu_vitals_4h/first_ranked_...,121,08f34fcf6af1c159c2611abf905170ab729b7d0ad9c5e8...


## Result

The project now has a reproducible source-to-resource derivation:

**MIMIC-IV Demo v2.2 → first-4-hour HR/RR/SpO₂ cohort → linked five-table source-faithful resource.**

This bundle is the input to the semantic FAIRification step. The contrast is deliberate: the files above show what the bounded project object looks like **before** FHIR modelling and terminology enrichment.
